# How to search SEC filings by keyword in Python

Keyword and phrase search across SEC filings from Python with requests and pandas: paging, facets, retries on 429 and writing the results to CSV.

Read it with charts at https://www.quantisnow.com/sec-api/guides/search-sec-filings-python. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

This is the whole loop: send a query, page through the matches, put them in a DataFrame. It uses `requests` and `pandas` and an API key in `QUANTISNOW_API_KEY`; a [new key](https://www.quantisnow.com/sec-api/keys) comes with free calls to try it.

## A search client in 30 lines

In [ ]:
import os, re, time, requests
import pandas as pd

URL = "https://www.quantisnow.com/api/v1/full-text"
AUTH = {"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"}

def search(query, **params):
    """Every match for a query, newest first. Raises past 10,000 matches."""
    out, start = [], 0
    while True:
        res = requests.post(URL, headers=AUTH, json={
            "query": query, "sort": "filedAt:desc", "from": start, "size": 50, **params,
        })
        if res.status_code == 429 and res.json()["error"]["code"] == "rate_limited":
            time.sleep(int(res.headers.get("Retry-After", "1")))
            continue
        res.raise_for_status()
        page = res.json()
        if page["total"]["relation"] == "gte":
            raise ValueError("over 10,000 matches; add a filedAt range")
        out += page["data"]
        start += 50
        if start >= page["total"]["value"]:
            return out

def frame(hits):
    return pd.DataFrame([{
        "filed": h["filedAt"][:10], "form": h["formType"], "ticker": h["ticker"],
        "company": h["companyName"], "items": ",".join(h["items"]),
        "passage": re.sub(r"</?em>", "", h["highlights"][0]) if h["highlights"] else "",
        "url": h["linkToFilingIndex"],
    } for h in hits])

## Run a search

In [ ]:
df = frame(search('"share repurchase program" AND (authorized OR approved)', formTypes=["8-K"]))
df.to_csv("buybacks.csv", index=False)
print(df.head())

### Which boards authorized a new or bigger share buyback?

Buyback announcements usually sit in the press release attached to an 8-K, which this searches along with the cover document.

```shell
curl -X POST https://www.quantisnow.com/api/v1/full-text \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "\"share repurchase program\" AND (authorized OR approved)",
  "formTypes": ["8-K"],
  "sort": "filedAt:desc",
  "size": 5
}'
```

Response (real, first 2 of 384):

```json
{
  "total": {
    "value": 384,
    "relation": "eq"
  },
  "took": 74,
  "data": [
    {
      "id": "0001628280-26-075523",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001628280-26-075523",
      "formType": "8-K",
      "filedAt": "2026-11-23T16:39:11-05:00",
      "cik": "0002011286",
      "companyName": "Amentum Holdings Inc.",
      "ticker": "AMTM",
      "tickers": [
        "AMTM"
      ],
      "items": [
        "2.02",
        "9.01"
      ],
      "description": "Amentum Holdings Inc. filed SEC Form 8-K: Results of Operations and Financial Condition, Financial Statements and Exhibits",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/2011286/000162828026075523/0001628280-26-075523-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/amentum-holdings-inc-filed-sec-form-8-k-results-of-operations-6804249",
      "score": null,
      "highlights": [
        "<em>Repurchase Program</em> CHANTILLY, VA., November 23, 2026 - Amentum Holdings, Inc.",
        "Combined with a disciplined capital allocation strategy, including our newly <em>authorized</em> $500 million <em>share repurchase program</em>, these priorities support our continued focus on creating long-term shareholder value.\"",
        "The <em>share repurchase program</em> provides additional flexibility to deploy capital as part of the Company's disciplined capital allocation strategy. 2 Backlog and Contract Awards As of October 2, 2026, the Company had total backlog"
      ]
    },
    {
      "id": "0001645590-26-000097",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001645590-26-000097",
      "formType": "8-K",
      "filedAt": "2026-11-23T16:12:08-05:00",
      "cik": "0001645590",
      "companyName": "Hewlett Packard Enterprise Company",
      "ticker": "HPE",
      "tickers": [
        "HPE"
      ],
      "items": [
        "2.02",
        "7.01",
        "9.01"
      ],
      "description": "Hewlett Packard Enterprise Company filed SEC Form 8-K: Results of Operations and Financial Condition, Regulation FD Disclosure, Financial Statements and Exhibits",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1645590/000164559026000097/0001645590-26-000097-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/hewlett-packard-enterprise-company-filed-sec-form-8-k-results-of-6803913",
      "score": null,
      "highlights": [
        "HPE's Board of Directors <em>authorized</em> an additional $5 billion in capacity for repurchases of common stock, bringing the total share repurchase authorization to approximately $7.3 billion as of November 23, 2026.",
        ": | 7.625% Series C mandatory convertible preferred stock, $0.01 par value (30 shares issued and outstanding as of October 31, 2026 and October 31, 2025, respectively) - | - | Common stock, $0.01 par value (9,600 shares <em>authorized</em>",
        "Share Repurchase Authorization Increase On November 23, 2026, HPE announced that its Board of Directors <em>approved</em> an increase of $5 billion in the amount <em>authorized</em> under HPE's <em>share repurchase program</em>."
      ]
    }
  ]
}
```

## Count before you page

`facets: true` with `size: 1` costs one call and tells you how a phrase splits by form, month and company. It is the cheap way to check a query catches what you meant before pulling every match.

In [ ]:
res = requests.post(URL, headers=AUTH, json={"query": '"GLP-1"', "formTypes": ["10-Q"], "size": 1, "facets": True}).json()
print(res["total"]["value"])
print(pd.DataFrame(res["facets"]["companyName"]))

### Which companies mention GLP-1 drugs in their 10-Q?

A theme screen: drug makers, but also food, medical device and insurance companies explaining what the drugs do to their business.

```shell
curl -X POST https://www.quantisnow.com/api/v1/full-text \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "\"GLP-1\"",
  "formTypes": ["10-Q"],
  "size": 5,
  "facets": true
}'
```

Response (real, first 2 of 49):

```json
{
  "total": {
    "value": 49,
    "relation": "eq"
  },
  "took": 138,
  "data": [
    {
      "id": "0001193125-26-458318",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001193125-26-458318",
      "formType": "10-Q",
      "filedAt": "2026-11-12T16:05:39-05:00",
      "cik": "0001572616",
      "companyName": "Fractyl Health Inc.",
      "ticker": "GUTS",
      "tickers": [
        "GUTS"
      ],
      "items": [],
      "description": "SEC Form 10-Q filed by Fractyl Health Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1572616/000119312526458318/0001193125-26-458318-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-q-filed-by-fractyl-health-inc-6787785",
      "score": 9.44,
      "highlights": [
        ". • The REVEAL-1 Cohort (n=22) is an open-label study in individuals with obesity who have lost at least 15% of their total body weight on a <em>GLP</em>-<em>1</em> medication and who either need or choose to discontinue <em>GLP</em>-<em>1</em> based therapy",
        "This gene therapy approach enables physiologic <em>GLP</em>-<em>1</em> secretion without the high circulating levels that contribute to side effects seen with systemic <em>GLP</em>-<em>1</em> based drugs.",
        "The Company was developing RJVA-002, a dual GIP/<em>GLP</em>-<em>1</em> gene therapy."
      ]
    },
    {
      "id": "0001493152-26-050097",
      "kind": "filing",
      "source": "SEC",
      "accessionNo": "0001493152-26-050097",
      "formType": "10-Q",
      "filedAt": "2026-11-05T09:30:48-05:00",
      "cik": "0001535955",
      "companyName": "Lipocine Inc.",
      "ticker": "LPCN",
      "tickers": [
        "LPCN"
      ],
      "items": [],
      "description": "SEC Form 10-Q filed by Lipocine Inc.",
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1535955/000149315226050097/0001493152-26-050097-index.htm",
      "quantisnowUrl": "https://www.quantisnow.com/insight/sec-form-10-q-filed-by-lipocine-inc-6775245",
      "score": 9.19,
      "highlights": [
        "in regulation of <em>GLP</em>-<em>1</em> through: ● | Enhancement of <em>GLP</em>-<em>1</em>-mediated insulin release from β cells through genomic- and non-genomic mechanisms | ● | Increase in <em>GLP</em>-<em>1</em> Receptor Expression in diabetics and non-diabetics | ● |",
        "In addition, 43% of <em>GLP</em>-<em>1</em> users lost ≥10% Stair Climb Power from baseline; the equivalent of almost eight years of expected age-related stair climb power loss was observed in just 4 months of <em>GLP</em>-<em>1</em> use.",
        "Elderly and sarcopenic <em>GLP</em>-<em>1</em> agonist users are the population of <em>GLP</em>-<em>1</em> users who are most vulnerable to accelerated lean mass loss and functional decline."
      ]
    }
  ],
  "facets": {
    "formType": [
      {
        "value": "10-Q",
        "count": 49
      }
    ],
    "month": [
      {
        "value": "2026-11",
        "count": 42
      },
      {
        "value": "2026-10",
        "count": 7
      }
    ],
    "companyName": [
      {
        "value": "Aardvark Therapeutics Inc.",
        "count": 1
      },
      {
        "value": "Alaunos Therapeutics Inc.",
        "count": 1
      },
      {
        "value": "Altimmune Inc.",
        "count": 1
      },
      {
        "value": "Amylyx Pharmaceuticals Inc.",
        "count": 1
      },
      {
        "value": "AptarGroup Inc.",
        "count": 1
      },
      {
        "value": "Apyx Medical Corporation",
        "count": 1
      },
      {
        "value": "Artivion Inc.",
        "count": 1
      },
      {
        "value": "Biomea Fusion Inc.",
        "count": 1
      },
      {
        "value": "Cardinal Health Inc.",
        "count": 1
      },
      {
        "value": "DarioHealth Corp.",
        "count": 1
      }
    ],
    "source": [
      {
        "value": "SEC",
        "count": 49
      }
    ]
  }
}
```

Each request is one call against your plan whatever its `size`, so always ask for 50 when paging. Plans allow from 2 to 50 requests a second, and the 429 handling above already respects the limit.